# 🚗 VN License Plate Recognition - Training Pipeline

Hướng dẫn train Detector (YOLOv8) và OCR (TrOCR) trên Google Colab

**Author:** VNPLPR Team  
**Date:** 2026-05-27

## 1. Setup Environment

In [ ]:
# Clone repository
!git clone https://github.com/your-username/ComputerVisionNew.git 2>/dev/null || echo "Repo already cloned or using local files"
%cd ComputerVisionNew

# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q ultralytics transformers datasets jiwer albumentations
!pip install -q torch torchvision --upgrade

# Verify installations
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
import os
OUTPUT_DIR = '/content/drive/MyDrive/vnplpr_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Models will be saved to: {OUTPUT_DIR}")

## 3. Upload Dataset

**Cách 1:** Upload trực tiếp lên Colab
```
Tải folder `data/` lên và upload vào Colab
```

**Cách 2:** Copy từ Google Drive
```python
!cp -r /content/drive/MyDrive/vnplpr_data/data.zip .
!unzip -q data.zip
```

In [ ]:
# Upload data manually using files panel
# Or use the following to upload
from google.colab import files

# Upload data.zip
uploaded = files.upload()

# Extract
!unzip -q data.zip -d . 2>/dev/null || echo "No data.zip found, using existing data"

---

# PHẦN A: TRAIN DETECTOR (YOLOv8)

## A1. Import và Setup

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path('/content/ComputerVisionNew')
sys.path.insert(0, str(PROJECT_ROOT))

# Verify data exists
data_yaml = PROJECT_ROOT / 'data/augmented_full/data.yaml'
if data_yaml.exists():
    print(f"✅ Data YAML found: {data_yaml}")
else:
    print("⚠️ Data not found. Please upload dataset first.")

## A2. Train Detector

In [ ]:
from ultralytics import YOLO
import torch

# Set device
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Training on device: {device}")

# Start training
model = YOLO('yolov8n.pt')  # Load pretrained YOLOv8n

results = model.train(
    data=str(PROJECT_ROOT / 'data/augmented_full/data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project=str(PROJECT_ROOT / 'runs/detect'),
    name='colab_detector',
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    patience=10,
    save=True,
    plots=True,
    verbose=True
)

In [ ]:
# Check results
import glob

best_model = glob.glob(f'{PROJECT_ROOT}/runs/detect/colab_detector/weights/best.pt')
if best_model:
    print(f"✅ Detector trained successfully!")
    print(f"Best model: {best_model[0]}")

    # Copy to Google Drive
    !cp {best_model[0]} /content/drive/MyDrive/vnplpr_models/detector_best.pt
    print("✅ Model saved to Google Drive")
else:
    print("❌ Training may have failed")

---

# PHẦN B: TRAIN OCR (TrOCR)

## B1. Export Plate Crops

In [ ]:
# Export crops using trained detector
!python scripts/export_crops_from_labels.py \
    --labels-dir data/augmented_full/labels \
    --images-dir data/augmented_full/images \
    --output-dir data/trocr_crops \
    --detector-model runs/detect/colab_detector/weights/best.pt

# Check output
!wc -l data/trocr_crops/train.csv

## B2. Train TrOCR

In [ ]:
%%time

# Train TrOCR
!python scripts/train_trocr.py \
    --train-csv data/trocr_crops/train.csv \
    --output-dir experiments/trocr_colab \
    --hyperparams-json configs/trocr/custom_plate.json

print("\n" + "="*50)
print("Training completed!")
print("="*50)

In [ ]:
# Copy TrOCR model to Google Drive
!cp -r experiments/trocr_colab /content/drive/MyDrive/vnplpr_models/
print("✅ TrOCR model saved to Google Drive")

# List saved models
!ls -lh /content/drive/MyDrive/vnplpr_models/

---

# PHẦN C: EVALUATION

## C1. Evaluate OCR

In [ ]:
# Evaluate OCR on test set
!python scripts/evaluate_ocr.py \
    --test-csv data/trocr_crops/train.csv \
    --ocr-model experiments/trocr_colab

# Check results
import pandas as pd
results = pd.read_csv('experiments/ocr_results.csv')
correct = (results['correct'] == 'Y').sum()
total = len(results)
print(f"\n📊 OCR Accuracy: {correct}/{total} = {correct/total*100:.1f}%")

---

# PHẦN D: DOWNLOAD MODELS

In [ ]:
# Download all models as zip
!zip -r /content/vnplpr_models.zip /content/drive/MyDrive/vnplpr_models/

# Download to local
from google.colab import files
files.download('/content/vnplpr_models.zip')

print("\n✅ All models downloaded!")

---

## 📝 Ghi chú

1. **Runtime:** Chọn GPU (T4 hoặc A100) từ Runtime → Change runtime type
2. **Colab Pro:** Nếu cần train lâu, dùng Colab Pro để tránh disconnect
3. **Data:** Upload dataset lên Google Drive trước để tránh mất khi disconnect
4. **Backup:** Models được auto-save vào Google Drive